# Zomato EDA

## Aryan Pathania (Ar3missss)

### Dataset: Zomato Restaurant Dataset (Zomato_Dataset.csv)


## Objectives

- Which cities have the most restaurants listed on Zomato?
- What are the most popular cuisines across all restaurants?
- How are restaurants distributed across different price ranges?
- How many restaurants offer online delivery and table booking?
- What is the distribution of restaurant ratings?
- Which price range gets the highest average rating?
- Which cuisines have the highest average ratings?


# Import Libraries


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Load Dataset


In [ ]:
file_path = os.path.join('data', 'Zomato_Dataset.csv')
df = pd.read_csv(file_path, encoding='latin1')
df.head()


# Data Overview

- Dataset contains 9551 rows and 17 columns
- Columns include restaurant details like city, cuisines, rating, votes, price range, online delivery and table booking availability
- Most columns are of object (categorical) type while RestaurantID, CountryCode, Price_range, Votes, Average_Cost_for_two and Rating are numerical
- Only 9 missing values are present in the Cuisines column
- No duplicate rows found
- The majority of restaurants in this dataset are from India


In [ ]:
df.duplicated().sum()


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.dtypes


# Data Cleaning and Preprocessing


### Missing Values


In [ ]:
df.isnull().sum().sort_values(ascending=False)


- Only the Cuisines column has 9 missing values
- These rows will be dropped since cuisine is a key column used in multiple analysis questions


### Handling Missing Values


In [ ]:
df = df[df['Cuisines'].notna()]
df.isnull().sum()


- The 9 rows with missing Cuisines were dropped since they were very few and cuisine is needed for genre-level analysis


### Price Range Labels


In [ ]:
price_map = {1: 'Cheap', 2: 'Affordable', 3: 'Expensive', 4: 'Very Expensive'}
df['Price_Label'] = df['Price_range'].map(price_map)
df[['Price_range', 'Price_Label']].head()


- Price range numbers were mapped to readable labels for easier interpretation in charts


# Feature Extraction


In [ ]:
df_cuisines = df.copy()
df_cuisines['Cuisines_split'] = df_cuisines['Cuisines'].str.split('|')
df_cuisines = df_cuisines.explode('Cuisines_split')
df_cuisines['Cuisines_split'] = df_cuisines['Cuisines_split'].str.strip()
df_cuisines[['RestaurantName', 'Cuisines', 'Cuisines_split']].head()


- Some restaurants serve multiple cuisines listed with a pipe separator, so the Cuisines column was split and exploded into individual cuisine rows for cuisine-level analysis


# Exploratory Data Analysis


## Q1 -- Which cities have the most restaurants listed on Zomato?


In [ ]:
city_counts = df['City'].value_counts().reset_index()
city_counts.columns = ['City', 'Count']
top_10_cities = city_counts.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_10_cities['City'], top_10_cities['Count'], color='red', edgecolor='white')
ax.set_title('Top 10 Cities with Most Restaurants on Zomato')
ax.set_xlabel('Number of Restaurants')
ax.set_ylabel('City')
ax.set_facecolor('#221F1F')
ax.bar_label(bars, fmt='%d', padding=1, color='white')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('outputs/zomato/Top 10 Cities with Most Restaurants.png', dpi=300)
plt.show()


- New Delhi has by far the most restaurants listed on Zomato, making it the most represented city in the dataset
- Gurgaon and Noida follow in second and third place, both being cities in the Delhi NCR region
- The dataset is heavily dominated by Indian cities, reflecting Zomato's origin and primary market


## Q2 -- What are the most popular cuisines across all restaurants?


In [ ]:
cuisine_counts = df_cuisines['Cuisines_split'].value_counts().reset_index()
cuisine_counts.columns = ['Cuisine', 'Count']
top_10_cuisines = cuisine_counts.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_10_cuisines['Cuisine'], top_10_cuisines['Count'], color='red', edgecolor='white')
ax.set_title('Top 10 Most Popular Cuisines on Zomato')
ax.set_xlabel('Count')
ax.set_ylabel('Cuisine')
ax.set_facecolor('#221F1F')
ax.bar_label(bars, fmt='%d', padding=1, color='white')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('outputs/zomato/Top 10 Most Popular Cuisines.png', dpi=300)
plt.show()


- North Indian cuisine is the most popular by a large margin, which aligns with the dataset being India-heavy
- Chinese and Fast Food are also very common, showing a preference for quick and familiar food options
- Cafe and Bakery appear in the top 10, indicating a growing coffee and dessert culture


## Q3 -- How are restaurants distributed across different price ranges?


In [ ]:
price_order = ['Cheap', 'Affordable', 'Expensive', 'Very Expensive']
price_counts = df['Price_Label'].value_counts().reindex(price_order).reset_index()
price_counts.columns = ['Price_Label', 'Count']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(price_counts['Price_Label'], price_counts['Count'], color='red', edgecolor='white')
ax.set_title('Restaurant Distribution by Price Range')
ax.set_xlabel('Price Range')
ax.set_ylabel('Number of Restaurants')
ax.set_facecolor('#221F1F')
ax.bar_label(bars, fmt='%d', padding=1, color='white')

plt.tight_layout()
plt.savefig('outputs/zomato/Restaurant Distribution by Price Range.png', dpi=300)
plt.show()


- The majority of restaurants fall in the Cheap and Affordable price ranges, showing that most listings are budget friendly
- Very Expensive restaurants are the least common, which is expected as premium dining is less common overall


## Q4 -- How many restaurants offer online delivery and table booking?


In [ ]:
delivery_counts = df['Has_Online_delivery'].value_counts().reset_index()
delivery_counts.columns = ['Has_Online_delivery', 'Count']

booking_counts = df['Has_Table_booking'].value_counts().reset_index()
booking_counts.columns = ['Has_Table_booking', 'Count']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars1 = axes[0].bar(delivery_counts['Has_Online_delivery'], delivery_counts['Count'], color=['red', '#221F1F'], edgecolor='white')
axes[0].set_title('Online Delivery Availability')
axes[0].set_xlabel('Has Online Delivery')
axes[0].set_ylabel('Number of Restaurants')
axes[0].bar_label(bars1, fmt='%d', padding=1)

bars2 = axes[1].bar(booking_counts['Has_Table_booking'], booking_counts['Count'], color=['red', '#221F1F'], edgecolor='white')
axes[1].set_title('Table Booking Availability')
axes[1].set_xlabel('Has Table Booking')
axes[1].set_ylabel('Number of Restaurants')
axes[1].bar_label(bars2, fmt='%d', padding=1)

plt.tight_layout()
plt.savefig('outputs/zomato/Online Delivery and Table Booking.png', dpi=300)
plt.show()


- The majority of restaurants do not offer online delivery, with only around 2451 out of 9542 providing this service
- Table booking is even less common, with only 1158 restaurants offering it
- This suggests that most listed restaurants are small or local places that rely on walk-in customers


## Q5 -- What is the distribution of restaurant ratings?


In [ ]:
average_rating = df['Rating'].mean()
median_rating = df['Rating'].median()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['Rating'], bins=30, color='red', edgecolor='white')
ax.set_title('Distribution of Restaurant Ratings on Zomato')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Restaurants')
ax.set_facecolor('#221F1F')
ax.axvline(average_rating, color='yellow', linestyle='--', linewidth=2, label=f'Mean: {average_rating:.1f}')
ax.axvline(median_rating, color='cyan', linestyle='-', linewidth=2, label=f'Median: {median_rating:.1f}')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/zomato/Distribution of Restaurant Ratings.png', dpi=300)
plt.show()


- The rating distribution is left-skewed, with a large spike at the lower end around 2.5
- The mean rating is around 2.9 and the median is 3.2, indicating that most restaurants have average to below average ratings
- Very few restaurants achieve ratings above 4.5, showing that high ratings are rare on the platform


## Q6 -- Which price range gets the highest average rating?


In [ ]:
price_order = ['Cheap', 'Affordable', 'Expensive', 'Very Expensive']
price_rating = df.groupby('Price_Label')['Rating'].mean().reset_index()
price_rating.columns = ['Price_Label', 'Avg_Rating']
price_rating['Price_Label'] = pd.Categorical(price_rating['Price_Label'], categories=price_order, ordered=True)
price_rating = price_rating.sort_values('Price_Label')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(price_rating['Price_Label'], price_rating['Avg_Rating'], color='red', edgecolor='white')
ax.set_title('Average Rating by Price Range')
ax.set_xlabel('Price Range')
ax.set_ylabel('Average Rating')
ax.set_facecolor('#221F1F')
ax.bar_label(bars, fmt='%.2f', padding=1, color='white')
ax.set_ylim(0, 5)

plt.tight_layout()
plt.savefig('outputs/zomato/Average Rating by Price Range.png', dpi=300)
plt.show()


- There is a clear positive relationship between price range and average rating
- Very Expensive restaurants have the highest average ratings, while Cheap restaurants have the lowest
- This suggests that customers tend to rate higher-end restaurants more favourably, possibly due to better food quality and service


## Q7 -- Which cuisines have the highest average ratings?


In [ ]:
cuisine_rating = df_cuisines.groupby('Cuisines_split')['Rating'].mean().reset_index()
cuisine_rating.columns = ['Cuisine', 'Avg_Rating']

cuisine_vote_counts = df_cuisines['Cuisines_split'].value_counts().reset_index()
cuisine_vote_counts.columns = ['Cuisine', 'Count']

cuisine_rating = cuisine_rating.merge(cuisine_vote_counts, on='Cuisine')
cuisine_rating = cuisine_rating[cuisine_rating['Count'] >= 50]
cuisine_rating = cuisine_rating.sort_values('Avg_Rating', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(cuisine_rating['Cuisine'], cuisine_rating['Avg_Rating'], color='red', edgecolor='white')
ax.set_title('Top 10 Cuisines by Average Rating (min 50 restaurants)')
ax.set_xlabel('Average Rating')
ax.set_ylabel('Cuisine')
ax.set_facecolor('#221F1F')
ax.bar_label(bars, fmt='%.2f', padding=1, color='white')
ax.invert_yaxis()
ax.set_xlim(0, 5)

plt.tight_layout()
plt.savefig('outputs/zomato/Top 10 Cuisines by Average Rating.png', dpi=300)
plt.show()


- Only cuisines with at least 50 restaurants were included to make the comparison more reliable
- Continental, Seafood and Mughlai cuisines tend to have the highest average ratings
- Street Food and Fast Food, despite being very popular, tend to score lower in average ratings


## Conclusion

This analysis gave a good overall picture of the Zomato restaurant dataset and what the platform looks like in terms of restaurant distribution, cuisines, pricing and ratings.

The dataset is heavily India-focused, with New Delhi alone accounting for more than half the total listings. Gurgaon and Noida follow, both being part of the Delhi NCR area. This makes sense given that Zomato was founded in India and built its initial user base there.

North Indian cuisine dominates the platform by a large margin, followed by Chinese and Fast Food. This reflects the food preferences of the Indian market. Cafes and bakeries are also quite common, showing a growing trend in casual dining and coffee culture.

Most restaurants are listed in the Cheap and Affordable price ranges, meaning the platform largely caters to everyday budget dining. Very expensive restaurants are a small minority.

Online delivery and table booking are not very common across the platform. Most restaurants do not offer either service, which suggests that a large portion of listings are small local places that depend on walk-in customers.

The rating distribution is interesting. Most restaurants have ratings between 2.5 and 3.5, with very few crossing 4.5. There is also a clear trend where more expensive restaurants tend to get higher ratings, which could be because of better food quality, ambience and service.

Overall, Zomato in this dataset represents a budget-friendly, India-centric platform with a strong base in North Indian food and a customer base that is mostly visiting restaurants in person rather than ordering online.
